# A3.3 · PCA via SVD, the covariance eigendecomposition, and Eckart–Young

**ID:** A3.3 · **Target:** 15 min · **Plan week:** 2 · **Part:** A

**The drill.** PCA two ways: SVD of the centered data and `eigh` of the covariance. Then explained-variance ratios,
projection and reconstruction, and agreement with `sklearn.decomposition.PCA` **up to the sign** of each component.
Finally, check the Eckart–Young theorem numerically: the best rank-$k$ approximation error in Frobenius norm is
$\sqrt{\sum_{i>k}\sigma_i^2}$, and in spectral norm it is $\sigma_{k+1}$.
Guide A3: "PCA via SVD" (blank-editor classic, under 20 min).

**Rules.** Torch only inside your functions (`torch.linalg.svd`, `torch.linalg.eigh`). Float64. Components are rows
(like sklearn's `components_`, shape `(k, D)`).

## Sources
- C. Eckart, G. Young, *The approximation of one matrix by another of lower rank*, Psychometrika 1(3):211-218, 1936,
  DOI 10.1007/BF02288367 — https://doi.org/10.1007/BF02288367
- J. Shlens, *A Tutorial on Principal Component Analysis*, 2014, arXiv:1404.1100 — https://arxiv.org/abs/1404.1100
  (PCA via SVD vs covariance eigendecomposition).
- scikit-learn `PCA` (centering, `explained_variance_` with $n-1$, `explained_variance_ratio_`, sign convention via
  `svd_flip`): https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
- `torch.linalg.svd` / `torch.linalg.eigh` (ascending eigenvalues):
  https://docs.pytorch.org/docs/stable/generated/torch.linalg.svd.html ·
  https://docs.pytorch.org/docs/stable/generated/torch.linalg.eigh.html
- N. Halko, P.-G. Martinsson, J. A. Tropp, *Finding structure with randomness: Probabilistic algorithms for constructing
  approximate matrix decompositions*, SIAM Review 53(2):217-288, 2011, arXiv:0909.4061 — https://arxiv.org/abs/0909.4061
- scikit-learn `load_digits` (bundled): https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html

In [ ]:
import torch
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

from prep import Session, check, check_close, seed_everything

s = Session("A3.3", target_min=15)
seed_everything(0)
dd = dict(dtype=torch.float64)
X = torch.tensor(load_digits().data, **dd)          # (1797, 64)
n, D = X.shape
k = 10

## 1. PCA via SVD
Center: $X_c = X - \bar x$. Thin SVD $X_c = U S V^\top$. The principal directions are the rows of $V^\top$; the variance
along direction $i$ is $\sigma_i^2/(n-1)$; the explained-variance ratio is $\sigma_i^2/\sum_j\sigma_j^2$. Return
`(mean (D,), components (k, D), explained_variance (k,), ratio (k,))`.

In [ ]:
def pca_svd(X, k):
    raise NotImplementedError  # YOUR CODE HERE


def align_signs(A, B):
    """Flip the rows of A so that each has a non-negative dot product with the matching row of B."""
    return A * torch.sign((A * B).sum(1, keepdim=True))


mean, comps, ev, evr = pca_svd(X, k)
sk = PCA(n_components=k, svd_solver="full").fit(X.numpy())
sk_comps = torch.tensor(sk.components_)
check_close("mean", mean, sk.mean_, atol=1e-10)
check_close("components (up to sign)", align_signs(comps, sk_comps), sk_comps, atol=1e-8)
check_close("explained_variance_", ev, sk.explained_variance_, atol=1e-8)
check_close("explained_variance_ratio_", evr, sk.explained_variance_ratio_, atol=1e-10)
check_close("components orthonormal", comps @ comps.T, torch.eye(k, **dd), atol=1e-10)
print("top-10 explained variance ratio:", [round(v, 3) for v in evr.tolist()], f"(cumulative {evr.sum().item():.3f})")

## 2. PCA via the covariance matrix
$\Sigma = X_c^\top X_c/(n-1)$ is symmetric, so use `eigh`. It returns eigenvalues in **ascending** order: sort them
descending. The eigenvalues equal $\sigma_i^2/(n-1)$. This is fine for small $D$, but it squares the condition number
(as with the normal equations in A3.1), and forming $\Sigma$ costs $O(nD^2)$.

In [ ]:
def pca_eig(X, k):
    raise NotImplementedError  # YOUR CODE HERE


comps_e, ev_e = pca_eig(X, k)
check_close("eig components == SVD components (up to sign)", align_signs(comps_e, comps), comps, atol=1e-6)
check_close("eig eigenvalues == explained variance", ev_e, ev, atol=1e-8)

## 3. Project and reconstruct
Scores $Z = (X - \bar x)V_k$ with shape $(n, k)$; reconstruction $\hat X = Z V_k^\top + \bar x$. The squared reconstruction error
$\|X - \hat X\|_F^2$ equals the variance left out, $(n-1)\sum_{i>k}\lambda_i$.

In [ ]:
def pca_transform(X, mean, comps):
    raise NotImplementedError  # YOUR CODE HERE


def pca_reconstruct(Z, mean, comps):
    raise NotImplementedError  # YOUR CODE HERE


Z = pca_transform(X, mean, comps)
check_close("transform == sklearn (up to sign)", Z * torch.sign((comps * sk_comps).sum(1)), sk.transform(X.numpy()), atol=1e-7)
Xhat = pca_reconstruct(Z, mean, comps)
check_close("reconstruct == sklearn inverse_transform", Xhat, sk.inverse_transform(sk.transform(X.numpy())), atol=1e-7)
_, _, ev_all, _ = pca_svd(X, D)
check_close("||X - Xhat||_F^2 == (n-1) * sum of dropped eigenvalues", ((X - Xhat) ** 2).sum(), (n - 1) * ev_all[k:].sum(), atol=1e-6)

## 4. Eckart–Young, numerically
For any matrix $A$ with singular values $\sigma_1\ge\sigma_2\ge\dots$, the truncated SVD $A_k = U_k S_k V_k^\top$ is the best rank-$k$
approximation in both norms: $\|A - A_k\|_F = \sqrt{\sum_{i>k}\sigma_i^2}$ and $\|A - A_k\|_2 = \sigma_{k+1}$. Write `best_rank_k`
and check both identities; then check that a few other rank-$k$ matrices do worse.

In [ ]:
def best_rank_k(A, k):
    raise NotImplementedError  # YOUR CODE HERE


torch.manual_seed(0)
A = torch.randn(40, 25, **dd) @ torch.diag(torch.logspace(0, -2, 25, **dd)) @ torch.randn(25, 25, **dd)
S = torch.linalg.svdvals(A)
for kk in (1, 5, 12):
    Ak = best_rank_k(A, kk)
    check("rank(A_k) == k", torch.linalg.matrix_rank(Ak).item() == kk)
    check_close(f"k={kk}: ||A - A_k||_F == sqrt(sum tail sigma^2)", torch.linalg.matrix_norm(A - Ak), S[kk:].pow(2).sum().sqrt(), atol=1e-9)
    check_close(f"k={kk}: ||A - A_k||_2 == sigma_(k+1)", torch.linalg.matrix_norm(A - Ak, ord=2), S[kk], atol=1e-9)
    best = torch.linalg.matrix_norm(A - Ak).item()
    Qs = [torch.linalg.qr(torch.randn(40, kk, **dd)).Q for _ in range(5)]   # random rank-k projections Q Q^T A
    others = [torch.linalg.matrix_norm(A - Q @ (Q.T @ A)).item() for Q in Qs]
    check(f"k={kk}: random rank-k projections do worse", min(others) > best, f"{min(others):.3f} <= {best:.3f}")

## Say it out loud
1. Why must you center the data before PCA? What goes wrong if you don't?
2. SVD of $X_c$ vs `eigh` of the covariance: same answer? Which do you prefer, and when is the covariance route better?
3. Why are components only defined up to sign (and up to rotation when eigenvalues repeat)? How does sklearn fix the sign?
4. State Eckart–Young. How does it justify PCA as "best linear dimensionality reduction"?
5. How do you pick $k$? What does the explained-variance ratio miss?
6. Should you standardize features before PCA? What changes?

In [ ]:
s.finish(lookups=0, score=None, notes="")